In [3]:
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
import math

In [4]:
print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+xpu


In [ ]:
import os
# Force Level Zero to recognize the integrated GPU
os.environ["ZE_AFFINITY_MASK"] = "0"
# This helps the loader find the Intel runtime libraries you just installed
os.environ["PYTORCH_ENABLE_XPU_FALLBACK"] = "1"

In [ ]:
print("Is XPU available?", torch.xpu.is_available())

In [ ]:
import torch
print(f"Is XPU Compiled: {torch.xpu._is_compiled()}")

In [5]:
@dataclass
class GPTConfig:
    vocab_size: int = 50257
    block_size: int = 1024
    n_layer: int = 12
    n_head: int = 12   
    n_embd: int = 768
    # vocab_size: int = 65
    # block_size: int = 256
    # n_layer: int = 6
    # n_head: int = 6
    # n_embd: int = 384
    # dropout: float = 0.1

In [6]:
class CasualSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        self.n_head = config.n_head
        self.n_embed = config.n_embd
        #self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size)).view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embed, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q, k, v, 
            attn_mask=None, 
            dropout_p=self.dropout if self.training else 0, 
            is_causal=True
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y

In [7]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        #self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        #x = self.dropout(x)
        return x

In [8]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.attn = CasualSelfAttention(config)
        self.mlp = MLP(config)
        #self.attn = nn.MultiheadAttention(config.n_embd, config.n_head, dropout=config.dropout)
        # self.mlp = nn.Sequential(
        #     nn.Linear(config.n_embd, 4 * config.n_embd),
        #     nn.GELU(),
        #     nn.Linear(4 * config.n_embd, config.n_embd),
        #     nn.Dropout(config.dropout),
        # )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [9]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            #drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        std = 0.02
        if hasattr(module, "NANOGPT_SCALE_INIT"):
            std *= (2*self.config.n_layer) ** -0.5
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std) 

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config.block_size, "Cannot forward, model block size is exhausted."
        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)
        pos_emb = self.transformer.wpe(pos)
        tok_emb = self.transformer.wte(idx)
        x = pos_emb + tok_emb
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        #return logits
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

In [10]:
import tiktoken
class DataLoader(torch.utils.data.Dataset):
    def __init__(self, B, T, data=None):
        self.data = data if data is not None else self.load_data()
        self.B = B
        self.T = T
        self.tokens = torch.tensor(self.data)
        self.current_pos = 0
        print(f"DataLoader initialized with {len(self.tokens)} tokens.")
        print(f"1 epoch will have {len(self.tokens) // (B * T)} batches.")

    def load_data(self):
        with open("./data/input.txt", "r") as f:
            text = f.read()
        return tiktoken.get_encoding("gpt2").encode(text)

    def getBatch(self):
        B, T = self.B, self.T
        buff = self.tokens[self.current_pos:self.current_pos+B*T+1]
        x = buff[:-1].view(B, T)
        y = buff[1:].view(B, T)
        self.current_pos += B*T
        if self.current_pos + B*T + 1 >= len(self.tokens):
            self.current_pos = 0
        return x, y

In [16]:
import os

# 1. Define the base paths based on your installation
vs_base = r"C:\Program Files (x86)\Microsoft Visual Studio\18\BuildTools\VC\Tools\MSVC\14.51.36014"
sdk_base = r"C:\Program Files (x86)\Windows Kits\10"
sdk_ver = "10.0.26100.0"

# 2. Update INCLUDE (This fixes the 'crtdbg.h' not found error)
existing_include = os.environ.get("INCLUDE", "")
new_includes = [
    os.path.join(vs_base, "include"),
    os.path.join(sdk_base, "Include", sdk_ver, "ucrt"),
    os.path.join(sdk_base, "Include", sdk_ver, "shared"),
    os.path.join(sdk_base, "Include", sdk_ver, "um"),
    os.path.join(sdk_base, "Include", sdk_ver, "winrt"),
]
os.environ["INCLUDE"] = ";".join(new_includes) + ";" + existing_include

# 3. Update LIB (This prevents 'link' errors later)
existing_lib = os.environ.get("LIB", "")
new_libs = [
    os.path.join(vs_base, "lib", "x64"),
    os.path.join(sdk_base, "Lib", sdk_ver, "ucrt", "x64"),
    os.path.join(sdk_base, "Lib", sdk_ver, "um", "x64"),
]
os.environ["LIB"] = ";".join(new_libs) + ";" + existing_lib

print("Environment paths updated for MSVC and Windows SDK.")

Environment paths updated for MSVC and Windows SDK.


In [17]:
import torch
import os

import torch
import os
from torch._inductor import config

device = torch.device("cpu")

num_return_sequences = 5
max_length = 30
B = 4
T = 32

model = GPT(GPTConfig())
model = torch.compile(model)
model.eval()
train_loader = DataLoader(B=B, T=T)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
for i in range(20):
    x, y = train_loader.getBatch()
    x = x.to(device, non_blocking=True)
    y = y.to(device, non_blocking=True)
    optimizer.zero_grad()
    logits, loss = model(x, targets=y)
    loss.backward()
    optimizer.step()
    print(f"Step {i+1}, Loss: {loss.item()}")

DataLoader initialized with 338024 tokens.
1 epoch will have 2640 batches.
Step 1, Loss: 10.8677978515625
Step 2, Loss: 9.723618507385254
Step 3, Loss: 9.161078453063965
Step 4, Loss: 9.181286811828613
Step 5, Loss: 8.690096855163574
Step 6, Loss: 8.357491493225098
Step 7, Loss: 9.039762496948242
Step 8, Loss: 8.75549030303955
Step 9, Loss: 8.109833717346191
Step 10, Loss: 7.955645561218262
Step 11, Loss: 8.321834564208984
Step 12, Loss: 7.479214668273926
Step 13, Loss: 7.7557692527771
Step 14, Loss: 7.469951152801514
Step 15, Loss: 7.512796401977539
Step 16, Loss: 7.319540500640869
Step 17, Loss: 7.47870397567749
Step 18, Loss: 8.320402145385742
Step 19, Loss: 7.230023384094238
Step 20, Loss: 7.731740951538086


In [ ]:
import torch
print("PyTorch version:", torch.__version__)

In [ ]:
import torch

print("XPU available:", torch.xpu.is_available())
print("Torch version:", torch.__version__)